# Phase 4 — Wav2Vec2 Frozen Extraction (fresh kernel)

**Date:** 2026-05-07 · **Project:** DL-2 Deepfake Audio Detection · **Notebook:** 1 of 2 (extraction-only)

Phase 3 deferred experiments 3.4 / 3.5 because Wav2Vec2 frozen extraction hung > 40 min when
run in the same kernel as the librosa augmentation pipeline. A standalone process completes the
same workload in seconds. Phase 4's fix: extract W2V2 features in this isolated notebook,
save to disk, and let the main Phase 4 notebook stack them with the tuned XGBoost.

Subset matches Phase 3 exactly (seed=42, 500 train / 180 test / 100 hemg) so cross-phase
comparison is on the same clips.


In [1]:
import os, sys, time, json
from pathlib import Path

import numpy as np
import torch

PROJ = Path('..').resolve()
sys.path.insert(0, str(PROJ))
os.environ.setdefault('HF_DATASETS_CACHE', str(PROJ / 'data' / 'raw' / 'hf_cache'))
os.environ.setdefault('TRANSFORMERS_NO_ADVISORY_WARNINGS', '1')

print('cwd:', PROJ)
print('torch:', torch.__version__, 'mps:', torch.backends.mps.is_available())
DEVICE = torch.device('mps') if torch.backends.mps.is_available() else torch.device('cpu')
print('device:', DEVICE)


cwd: /Users/anthonyrodrigues/Desktop/YC-Portfolio-Projects/Deepfake-Audio-Detection
torch: 2.11.0 mps: True
device: mps


## 1. Load datasets — same protocol as Phase 3

We reproduce the Phase 3 split *exactly*: train = 500 from garystafford['train'], test = 180 from
garystafford['test'], hemg = 100 from Hemg['train'], all with seed=42.


In [2]:
from datasets import load_dataset

t0 = time.time()
gs = load_dataset('garystafford/deepfake-audio-detection', cache_dir=str(PROJ / 'data' / 'raw' / 'hf_cache'))
hg = load_dataset('Hemg/Deepfake-Audio-Dataset', cache_dir=str(PROJ / 'data' / 'raw' / 'hf_cache'))
print(f'loaded in {time.time()-t0:.1f}s')
print('garystafford:', {k: len(v) for k, v in gs.items()})
print('hemg:', {k: len(v) for k, v in hg.items()})


loaded in 7.9s
garystafford: {'train': 1866}
hemg: {'train': 100}


In [3]:
# Phase 3 subset protocol — pinned indices
TRAIN_N, TEST_N, HEMG_N = 500, 180, 100
SEED = 42

rng = np.random.default_rng(SEED)
gs_train_full = gs['train']
gs_test_full = gs['test'] if 'test' in gs else gs['train']  # fallback if no test split
hemg_full = hg[list(hg.keys())[0]]

train_idx = rng.choice(len(gs_train_full), size=min(TRAIN_N, len(gs_train_full)), replace=False)
test_idx = rng.choice(len(gs_test_full), size=min(TEST_N, len(gs_test_full)), replace=False)
hemg_idx = rng.choice(len(hemg_full), size=min(HEMG_N, len(hemg_full)), replace=False)

print(f'train: {len(train_idx)}, test: {len(test_idx)}, hemg: {len(hemg_idx)}')

# Save indices for cross-notebook reproducibility
idx_path = PROJ / 'results' / 'phase4_subset_idx.json'
with open(idx_path, 'w') as f:
    json.dump({
        'seed': SEED,
        'train_idx': train_idx.tolist(),
        'test_idx': test_idx.tolist(),
        'hemg_idx': hemg_idx.tolist(),
    }, f)
print('saved indices to', idx_path.relative_to(PROJ))


train: 500, test: 180, hemg: 100
saved indices to results/phase4_subset_idx.json


## 2. Load Wav2Vec2-base (frozen)

`facebook/wav2vec2-base` — 95M params, 768d hidden state. Frozen feature extractor: we mean-pool
the final hidden state across time, getting one 768d vector per clip. No fine-tuning. This is
the cheapest viable W2V2 protocol — anything more (full sequence, pooled by attention, etc.)
needs a fresh GPU run.


In [4]:
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor

t0 = time.time()
model_id = 'facebook/wav2vec2-base'
feat_ext = Wav2Vec2FeatureExtractor.from_pretrained(model_id)
w2v2 = Wav2Vec2Model.from_pretrained(model_id)
w2v2 = w2v2.to(DEVICE)
w2v2.eval()
for p in w2v2.parameters():
    p.requires_grad_(False)
print(f'loaded {model_id} in {time.time()-t0:.1f}s, hidden_size={w2v2.config.hidden_size}')


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


loaded facebook/wav2vec2-base in 2.9s, hidden_size=768


## 3. Extraction loop

Fixed 1.5 s clips at 16 kHz (matches Phase 3 hung-cell protocol). We process one clip at a time
to keep MPS memory predictable. Expected wall time: ~30–90 s per 100-clip set.


In [5]:
import librosa

TARGET_SR = 16000
DURATION_S = 1.5
TARGET_LEN = int(TARGET_SR * DURATION_S)

def to_fixed_mono(arr, sr):
    arr = np.asarray(arr, dtype=np.float32)
    if arr.ndim > 1:
        arr = arr.mean(axis=-1)
    if sr != TARGET_SR:
        arr = librosa.resample(arr, orig_sr=sr, target_sr=TARGET_SR)
    if arr.shape[0] > TARGET_LEN:
        arr = arr[:TARGET_LEN]
    elif arr.shape[0] < TARGET_LEN:
        arr = np.pad(arr, (0, TARGET_LEN - arr.shape[0]))
    return arr.astype(np.float32)


def extract_split(ds, idx_array, label_field='label', name=''):
    feats = np.zeros((len(idx_array), w2v2.config.hidden_size), dtype=np.float32)
    labels = np.zeros(len(idx_array), dtype=np.int64)
    t0 = time.time()
    for i, ex_i in enumerate(idx_array):
        ex = ds[int(ex_i)]
        a = ex['audio']
        y = to_fixed_mono(a['array'], int(a['sampling_rate']))
        x = feat_ext(y, sampling_rate=TARGET_SR, return_tensors='pt')['input_values'].to(DEVICE)
        with torch.no_grad():
            out = w2v2(x).last_hidden_state  # (1, T, 768)
        feats[i] = out.mean(dim=1).squeeze(0).cpu().numpy().astype(np.float32)
        labels[i] = int(ex[label_field])
        if (i + 1) % 25 == 0 or i == len(idx_array) - 1:
            print(f'  {name}: {i+1}/{len(idx_array)} in {time.time()-t0:.1f}s', flush=True)
    return feats, labels


print('extracting train...')
X_w2v_train, y_train = extract_split(gs_train_full, train_idx, name='train')
print('extracting test...')
X_w2v_test, y_test = extract_split(gs_test_full, test_idx, name='test')
print('extracting hemg...')
X_w2v_hemg, y_hemg = extract_split(hemg_full, hemg_idx, name='hemg')

print()
print('shapes:', X_w2v_train.shape, X_w2v_test.shape, X_w2v_hemg.shape)
print('label balance — train:', np.bincount(y_train), 'test:', np.bincount(y_test), 'hemg:', np.bincount(y_hemg))


extracting train...


  train: 25/500 in 3.0s


  train: 50/500 in 3.5s


  train: 75/500 in 4.1s


  train: 100/500 in 4.6s


...

  hemg: 25/100 in 0.5s


  hemg: 50/100 in 1.1s


  hemg: 75/100 in 1.6s


  hemg: 100/100 in 2.2s



shapes: (500, 768) (180, 768) (100, 768)
label balance — train: [246 254] test: [87 93] hemg: [50 50]


In [6]:
out_dir = PROJ / 'results'
np.save(out_dir / 'phase4_w2v2_train.npy', X_w2v_train)
np.save(out_dir / 'phase4_w2v2_test.npy', X_w2v_test)
np.save(out_dir / 'phase4_w2v2_hemg.npy', X_w2v_hemg)
np.save(out_dir / 'phase4_y_train.npy', y_train)
np.save(out_dir / 'phase4_y_test.npy', y_test)
np.save(out_dir / 'phase4_y_hemg.npy', y_hemg)
print('saved 6 arrays to', out_dir.relative_to(PROJ))
print('train w2v2:', X_w2v_train.shape, 'mean abs:', np.abs(X_w2v_train).mean())


saved 6 arrays to results
train w2v2: (500, 768) mean abs: 0.18399137


## 4. Sanity check — does W2V2 separate real vs fake at all in-domain?

A 5-fold logistic regression on the 768d frozen embeddings. If this is at chance, W2V2 isn't
giving us any signal and we should skip stacking.


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

pipe = Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression(max_iter=1000, C=1.0))])
cv = cross_val_score(pipe, X_w2v_train, y_train, cv=5, scoring='roc_auc')
print(f'5-fold ROC-AUC on garystafford train (W2V2 + LogReg): {cv.mean():.3f} ± {cv.std():.3f}')

pipe.fit(X_w2v_train, y_train)
proba_test = pipe.predict_proba(X_w2v_test)[:, 1]
proba_hemg = pipe.predict_proba(X_w2v_hemg)[:, 1]

from sklearn.metrics import roc_auc_score
print(f'in-domain test ROC-AUC: {roc_auc_score(y_test, proba_test):.3f}')
print(f'cross-domain Hemg ROC-AUC: {roc_auc_score(y_hemg, proba_hemg):.3f}')

np.save(out_dir / 'phase4_w2v2_lr_proba_test.npy', proba_test)
np.save(out_dir / 'phase4_w2v2_lr_proba_hemg.npy', proba_hemg)
print('saved W2V2+LogReg test/hemg probas')


5-fold ROC-AUC on garystafford train (W2V2 + LogReg): 0.992 ± 0.007
in-domain test ROC-AUC: 0.999
cross-domain Hemg ROC-AUC: 0.559
saved W2V2+LogReg test/hemg probas


## Done

Hand off to `phase4_tuning.ipynb`, which loads the saved arrays and does Optuna tuning + stacking
+ error analysis without ever touching librosa augmentation again (the thing that hung Phase 3).
